In [1]:

# Install first if needed:
# %pip install pandas numpy matplotlib scikit-learn nltk

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

nltk.download("vader_lexicon", quiet=True)

# 1. Load dataset
df = pd.read_csv("RY_TD_stocks.csv")
df.columns = df.columns.str.strip().str.lower()

print(df.head())
print(df.info())

# 2. K-Means clustering
price_col = next(
    (c for c in ["close", "adj close", "adj_close", "price"]
     if c in df.columns), None
)
if price_col is None:
    raise ValueError("No closing-price column found.")

df[price_col] = pd.to_numeric(df[price_col], errors="coerce")
df["return"] = df[price_col].pct_change()
df["volatility"] = df["return"].rolling(5).std()

features = df[["return", "volatility"]].replace(
    [np.inf, -np.inf], np.nan
).dropna()

if len(features) < 3:
    raise ValueError("Not enough valid stock data.")

X = StandardScaler().fit_transform(features)
k = min(3, len(features))
df.loc[features.index, "cluster"] = KMeans(
    n_clusters=k, random_state=42, n_init=10
).fit_predict(X)

# Visualise clusters
pca = PCA(n_components=2)
coords = pca.fit_transform(X)

plt.scatter(coords[:, 0], coords[:, 1],
            c=df.loc[features.index, "cluster"], cmap="viridis")
plt.title("K-Means Stock Clusters")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()

# 3. NLP sentiment analysis (requires a text column)
text_col = next(
    (c for c in ["headline", "news", "text", "description"]
     if c in df.columns), None
)

if text_col:
    sia = SentimentIntensityAnalyzer()
    df["sentiment"] = df[text_col].fillna("").astype(str).apply(
        lambda x: sia.polarity_scores(x)["compound"]
    )
    df["sentiment_label"] = df["sentiment"].apply(
        lambda x: "Positive" if x >= 0.05
        else "Negative" if x <= -0.05 else "Neutral"
    )
    print(df[[text_col, "sentiment", "sentiment_label"]].head())
else:
    print("No news/text column found. NLP requires textual data.")

# 4. Save results
df.to_csv("RY_TD_stocks_results.csv", index=False)
print("Done! Results saved to RY_TD_stocks_results.csv")


  ticker                    company exchange  share price (usd)  \
0     RY       Royal Bank of Canada     NYSE             191.91   
1     TD  The Toronto-Dominion Bank     NYSE             115.10   

  market cap (usd)  trailing p/e  eps (usd) 52-week range (usd)  \
0          266.71B         17.25      11.33       143.13-218.57   
1          194.46B         17.60       6.98        77.95-125.47   

                                         description  
0  Canada's largest bank by market capitalization...  
1  A major "Big Five" Canadian bank with a strong...  
<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ticker               2 non-null      str    
 1   company              2 non-null      str    
 2   exchange             2 non-null      str    
 3   share price (usd)    2 non-null      float64
 4   market cap (usd)     2 non-null      

ValueError: No closing-price column found.